In [ ]:
# @title Install required dependencies
!pip install -q "crewai[tools]>=1.8.0" python-dotenv ddgs pydantic pandas pypdf


In [ ]:
import os
import io
import pandas as pd
from pydantic import BaseModel, Field
from google.colab import userdata
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from crewai_tools import FileWriterTool

SAMPLE_SALES_CSV = """product,region,units_sold,unit_price,revenue
Cloud Backup,North America,120,50,6000
API Gateway,Europe,80,120,9600
Auth Service,North America,200,30,6000
Cloud Backup,Europe,90,50,4500
API Gateway,Asia-Pacific,110,120,13200
Auth Service,Europe,150,30,4500
Monitoring,North America,70,80,5600
Monitoring,Asia-Pacific,60,80,4800
"""

class SalesMetrics(BaseModel):
    total_revenue: float = Field(..., description="Sum of total revenue across all regions")
    top_product_by_revenue: str = Field(..., description="Product generating highest revenue")
    top_region_by_revenue: str = Field(..., description="Region generating highest revenue")
    total_units_sold: int = Field(..., description="Total units sold across all products")

try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    llm = LLM(
        model="google/gemini-3.1-flash-lite-preview",
        api_key=userdata.get("GEMINI_API_KEY"),
        temperature=0.2,
        max_tokens=1200,
    )

    with open("sales.csv", "w", encoding="utf-8") as f:
        f.write(SAMPLE_SALES_CSV.strip())

    @tool("ComputeSalesMetrics")
    def compute_sales_metrics(file_path: str = "sales.csv") -> str:
        """Reads sales.csv and calculates revenue and unit metrics."""
        df = pd.read_csv(file_path)
        total_rev = df["revenue"].sum()
        total_units = df["units_sold"].sum()
        top_prod = df.groupby("product")["revenue"].sum().idxmax()
        top_region = df.groupby("region")["revenue"].sum().idxmax()
        return (
            f"Total Revenue: {total_rev}\n"
            f"Total Units Sold: {total_units}\n"
            f"Top Product: {top_prod}\n"
            f"Top Region: {top_region}"
        )

    data_analyst = Agent(
        role="Data Analyst",
        goal="Calculate aggregate metrics from sales.csv",
        backstory="Quantitative analyst skilled at computing accurate financial and volume totals.",
        llm=llm,
        tools=[compute_sales_metrics],
        verbose=True,
        allow_delegation=False,
    )

    insight_writer = Agent(
        role="Business Insights Specialist",
        goal="Interpret calculated metrics and draft strategic observations",
        backstory="Business strategist transforming metrics into executive growth narratives.",
        llm=llm,
        verbose=True,
        allow_delegation=False,
    )

    reviewer = Agent(
        role="Executive Editor",
        goal="Format final report with recommendations and save to output/sales_report.md",
        backstory="Operations director verifying report clarity and data precision.",
        llm=llm,
        tools=[FileWriterTool()],
        verbose=True,
        allow_delegation=False,
    )

    metrics_task = Task(
        description="Use ComputeSalesMetrics on sales.csv to compute totals, top products, and regions.",
        expected_output="Computed numeric metrics for revenue, units, and regional leaders.",
        agent=data_analyst,
        output_pydantic=SalesMetrics,
    )

    insights_task = Task(
        description="Review calculated metrics and outline market trends and high-performing verticals.",
        expected_output="Bullet list of analytical insights based on sales figures.",
        agent=insight_writer,
        context=[metrics_task],
    )

    report_task = Task(
        description="Compile metrics and insights into a formal sales report. Save to output/sales_report.md.",
        expected_output="Complete sales report in Markdown.",
        agent=reviewer,
        context=[metrics_task, insights_task],
    )

    crew = Crew(
        agents=[data_analyst, insight_writer, reviewer],
        tasks=[metrics_task, insights_task, report_task],
        process=Process.sequential,
        verbose=True,
    )

    result = crew.kickoff()
    print("\n--- Sales Report ---\n")
    print(result.raw)

except Exception as e:
    print("Error:", e)
